# The leakage premium — master pipeline

Companion code for *"Leakage makes every problem look equally easy: a multi-dataset
measurement of the resampling-before-splitting artefact"* and its two companion papers.

This notebook runs the whole study end to end. Each step is a thin call into a module in
`Code/`, so the notebook and the batch scripts execute identical code.

| Step | Script | Output |
|---|---|---|
| 1. Build the corpus | `01_fetch_datasets.py` | `data/processed/*.npz`, `manifest.csv` |
| 2. Main factorial | `run_experiment.py` | `results/raw_shard*.csv` |
| 3. Controlled simulation | `run_simulation.py` | `results/simulation.csv` |
| 4. Mitigation sweep | `run_mitigation.py` | `results/mitigation.csv` |
| 5. Analysis | `03_analysis.py` | `results/summary.json`, tables, figures |
| 6. Export the screening model | `04_export_model.py` | `leakcheck/model.json` |
| 7. Build the manuscripts | `Paper2*/build_paper2*.py` | three `.docx` files |
| 8. Verify every number | `05_verify_numbers.py` | pass/fail |

**Runtime.** Step 2 is the expensive one. On two cores it takes a few hours; it shards
across processes and checkpoints after every dataset, so it can be interrupted and resumed.

In [ ]:
import sys, json, warnings
from pathlib import Path
warnings.filterwarnings('ignore')

CODE = Path.cwd()
while CODE.name != 'Code' and CODE.parent != CODE:
    CODE = CODE.parent
sys.path.insert(0, str(CODE))

import numpy as np, pandas as pd
import leakage_lib as L
pd.set_option('display.width', 200, 'display.max_columns', 60)
print('corpus  :', L.PROC)
print('results :', L.RES)

## 1. The corpus

In [ ]:
# Safe to re-run: datasets already present are re-derived deterministically.
!python "/root/p2/Code/01_fetch_datasets.py"

In [ ]:
man = L.manifest()
print(f'{len(man)} datasets   IR {man.imbalance_ratio.min():.2f} - {man.imbalance_ratio.max():.1f}'
      f'   n {man.n.min()} - {man.n.max()}   p {man.n_features.min()} - {man.n_features.max()}')
man.groupby('domain').agg(datasets=('name','size'), median_IR=('imbalance_ratio','median'),
                          total_rows=('n','sum'))

In [ ]:
man[['name','domain','n','n_features','n_minority','imbalance_ratio','construction']]

## 2. The main factorial

Three conditions that differ only in where the resampler sits:

| Condition | Procedure |
|---|---|
| `leaked`  | resample the **whole** dataset, then split 70:30 |
| `correct` | split 70:30, then resample the **training partition only** |
| `none`    | split 70:30, no resampling |

Run the shards from a terminal so they proceed in parallel — two shards on a two-core
machine, four on four cores. Each writes its own CSV and skips datasets already done.

```bash
for s in 0 1; do
    OMP_NUM_THREADS=1 python run_experiment.py --shard $s --nshards 2 --seeds 10 &
done
```

The cell below runs a single-process version instead, which is fine for a quick check with
fewer seeds but slow for the full grid.

In [ ]:
QUICK = True          # set False only if you intend to run the full grid in-notebook

if QUICK:
    demo = L.run_dataset('ibm_hr', clf_set=['LogReg','DecTree','RandForest'],
                         samp_set=['SMOTE'], n_seeds=2)
    print(demo.groupby('condition').f1.agg(['mean','std','size']).round(4))
else:
    import subprocess
    subprocess.run([sys.executable, str(CODE/'run_experiment.py'),
                    '--shard','0','--nshards','1','--seeds','10'], check=True)

## 3-4. Simulation and mitigation sweep

In [ ]:
# Both are independent of the main factorial and can run while it is still going.
# !python "/root/p2/Code/run_simulation.py" --seeds 3
# !python "/root/p2/Code/run_mitigation.py" --seeds 3
print('run these from a terminal; they take 30-90 minutes each')

## 5. Analysis — produces every table, figure and number in the papers

In [ ]:
!python "/root/p2/Code/03_analysis.py"

In [ ]:
S = json.load(open(L.RES/'summary.json'))
for k in ['n_datasets','n_paired','leaked_f1_mean','correct_f1_mean','variance_ratio',
          'mean_premium','law_r2','metareg_r2_correct','metareg_r2_leaked','diag_auc']:
    print(f'{k:24s} {S.get(k)}')

## 6-8. Screening model, manuscripts, verification

In [ ]:
!python "/root/p2/Code/04_export_model.py"
!python "/root/p2/Code/Paper2A/build_paper2A.py"
!python "/root/p2/Code/Paper2B/build_paper2B.py"
!python "/root/p2/Code/Paper2C/build_paper2C.py"
!python "/root/p2/Code/05_verify_numbers.py"

---
### Checklist before submission

- [ ] the full grid completed on every dataset in the manifest
- [ ] `summary.json` regenerated from that single run
- [ ] `05_verify_numbers.py` passes all three checks
- [ ] library versions in `summary.json['versions']` match the run being reported
- [ ] `raw_results.csv`, `manifest.csv` and the figures archived with a DOI